# Notebook 2 · Agir avec des outils

*Formation « Comprendre et développer des agents IA spécialisés », module 2.*

Les clients de **Djassa** posent chaque jour les mêmes questions : *« L'iPhone 16 rose est dispo ? »*, *« Vous avez les Air Force 1 en 42 ? »*, *« Je cherche un cadeau à moins de 40 000 FCFA, vous conseillez quoi ? »*, *« Je le reçois quand à Cocody ? Je peux le renvoyer ? »* On construit **Awa**, une conseillère de vente IA qui répond **en consultant le vrai catalogue** (stock, prix, avis clients) au lieu d'inventer.

<p align="center"><img src="assets/diagrams/nb2_mise_en_contexte.png" alt="Quatre questions de clients de Djassa : Awa, conseillère de vente IA, y répond en consultant le vrai catalogue (stock, prix, avis, livraison, retours) au lieu d'inventer" width="1000"></p>

C'est le passage de l'assistant à l'**agent** : outils, tool calling, boucle ReAct et validation des arguments. Chaque notion est d'abord montrée **en Python pur**, puis **avec LangChain ou LangGraph**.

<p align="center"><img src="assets/diagrams/nb2_parcours_agent.png" alt="De l'assistant à l'agent : outils (étape 3), tool calling (étape 4), boucle ReAct (étape 5), validation (étape 6), puis agent" width="1000"></p>

**Attention** : Ici, Awa **consulte** seulement (stock, prix, livraison, retours). Réserver, rembourser ou modifier une commande sont des actions **à risque** : elles exigent autorisation, contrôles métier et souvent une validation humaine (notebook 4).

<p align="center"><img src="assets/diagrams/nb2_perimetre_awa.png" alt="Dans ce notebook, Awa consulte stock, prix, livraison et retours ; réserver, rembourser et modifier une commande sont des actions à risque traitées au notebook 4" width="900"></p>

Exécute les cellules dans l'ordre (`Maj + Entrée`). Tout est dans ce notebook.

**Première lecture rapide ?** Tu peux sauter les sections « Avec LangChain » et « Avec LangGraph » : le code en Python pur contient déjà tout le mécanisme, et les frameworks ne font que l'automatiser.

---
## Étape 0 · Configuration

**Objectif** : choisir le modèle de langage (LLM) qui va animer Awa.

Tu as deux choix :
- **`"openai"`** (par défaut) : un modèle d'OpenAI via Internet. Il faut une clé `OPENAI_API_KEY` dans le fichier `.env`.
- **`"ollama"`** (option) : un modèle installé sur ton ordinateur. C'est gratuit et ça ne demande pas de clé, mais c'est moins fiable pour les agents.

Change seulement la première ligne de la cellule ci-dessous.

In [1]:
FOURNISSEUR = "openai"          # "openai" (clé dans .env) ou "ollama" (local, gratuit)

import json, os
from datetime import date, timedelta
from dotenv import load_dotenv
from openai import OpenAI

AUJOURDHUI = date(2026, 10, 27)  # date fixe du scénario : mêmes résultats pour tout le monde

load_dotenv()
if FOURNISSEUR == "ollama":     # Ollama parle le même « langage » que l'API OpenAI
    MODELE, BASE_URL, CLE = "qwen2.5:7b", "http://localhost:11434/v1", "ollama"
else:
    MODELE, BASE_URL, CLE = "gpt-5-mini", None, os.getenv("OPENAI_API_KEY")
REGLAGES = {"temperature": 0} if FOURNISSEUR == "ollama" else {}   # réponses plus stables en local

client = OpenAI(base_url=BASE_URL, api_key=CLE)
print(f"Prêt : {FOURNISSEUR} / {MODELE}")

Prêt : openai / gpt-5-mini


On prépare une petite fonction d'affichage pour que les traces de l'agent soient faciles à lire. Chaque ligne porte une étiquette : **Note** (un texte intermédiaire du LLM, rare), **Appelle** (l'agent appelle un outil), **Reçoit** (il reçoit le résultat de l'outil) et **Répond** (il répond au client).

In [2]:
ICONES = {"pensee": "Note", "outil": "Appelle", "resultat": "Reçoit", "reponse": "Répond"}

def afficher(etape, texte):
    """Affiche une ligne de trace, par exemple : Appelle │ verifier_stock(...)"""
    if etape == "reponse":
        print()
    print(f"{ICONES[etape]:<12} │ {texte}")

afficher("outil", "exemple d'affichage")

Appelle      │ exemple d'affichage


---
## Étape 1 · Le catalogue Djassa

**Objectif** : créer les données qu'Awa devra consulter.

Dans une vraie boutique, elles viendraient d'une base de données ou d'un logiciel e-commerce comme Shopify ou PrestaShop. Ici, on utilise de simples **dictionnaires Python** : c'est notre « mini base de données ».

Voici ce qu'on va construire :

<p align="center"><img src="assets/diagrams/nb2_catalogue_djassa.png" alt="Le catalogue Djassa : les informations de la boutique, 9 produits répartis en 3 rayons, et le détail d'un produit avec ses prix, sa fiche, son stock et ses avis" width="1000"></p>

On commence par les **informations générales** de la boutique :

In [3]:
BOUTIQUE = {
    "nom": "Djassa",
    "slogan": "Les bons plans du quotidien, livrés partout en Côte d'Ivoire",
    "livraison_gratuite_des": 50000,         # en FCFA, pour la livraison standard
    "service_client": "du lundi au samedi, de 8 h à 18 h, au +225 07 00 00 00 00 (appel ou WhatsApp)",
}
# Règles de livraison de la boutique (fictives, pour la démonstration) : (jours ouvrés, frais en FCFA)
ZONES = {
    "Abidjan":   {"villes": ["Abobo", "Adjamé", "Attécoubé", "Cocody", "Koumassi", "Marcory", "Plateau", "Port-Bouët",
                             "Treichville", "Yopougon"],
                  "standard": (2, 1500), "express": (1, 2500)},
    "Intérieur": {"villes": ["Bouaké", "Yamoussoukro", "San Pedro"],
                  "standard": (4, 3000), "express": (2, 5000)},
}
POLITIQUE_RETOUR = ("Retour sous 14 jours, article non utilisé et dans son emballage d'origine (gratuit en point relais Djassa). "
                    "Remboursement sous 5 jours ouvrés après réception du colis, sur le moyen de paiement utilisé.")
MOYENS_PAIEMENT = ["Orange Money", "MTN MoMo", "Wave", "carte bancaire", "paiement à la livraison (Abidjan uniquement)",
                   "paiement en 3 fois dès 150 000 FCFA"]

def fcfa(montant):
    """45000 -> '45 000 FCFA' : montants entiers, milliers séparés par une espace."""
    return f"{montant:,}".replace(",", " ") + " FCFA"

CATALOGUE = {}                               # rempli dans les 3 cellules suivantes

Puis les **produits**, rangés en 3 rayons. Chaque produit a la même structure : le prix (et l'ancien prix `prix_barre` s'il est en promotion), la fiche (`description`, `caracteristiques`), le `stock` (par couleur, ou par couleur **et** par taille pour les vêtements et les chaussures) et les avis (`note`, `nombre_avis`, `avis`).

**À savoir** : ce sont des cellules de **données** : pas besoin de tout lire, exécute-les simplement.

In [4]:
# Rayon HIGH-TECH
CATALOGUE["iphone 16"] = {
    "marque": "Apple", "categorie": "Téléphones", "prix": 690000, "prix_barre": 730000,
    "description": "Écran 6,1 pouces, puce A18, appareil photo 48 MP et USB-C. Idéal pour la photo et les réseaux sociaux.",
    "caracteristiques": {"stockage": "128 Go", "écran": "6,1 pouces", "photo": "48 MP", "garantie": "2 ans"},
    "stock": {"noir": 8, "blanc": 5, "rose": 0, "turquoise": 3},
    "note": 4.7, "nombre_avis": 328,
    "avis": [{"auteur": "Fatou", "note": 5, "texte": "Très bon téléphone, rapide et excellent en photo."},
             {"auteur": "Kouamé", "note": 4, "texte": "Très agréable à utiliser, mais le prix reste élevé."}],
}
CATALOGUE["samsung galaxy a36"] = {
    "marque": "Samsung", "categorie": "Téléphones", "prix": 245000, "prix_barre": 265000,
    "description": "Écran AMOLED 6,7 pouces, triple appareil photo et batterie qui tient la journée. Un très bon rapport qualité-prix.",
    "caracteristiques": {"stockage": "128 Go", "écran": "6,7 pouces", "photo": "50 MP", "garantie": "2 ans"},
    "stock": {"noir": 6, "bleu": 4, "argent": 2},
    "note": 4.5, "nombre_avis": 214,
    "avis": [{"auteur": "Salimata", "note": 5, "texte": "Écran magnifique et téléphone très fluide."},
             {"auteur": "Ibrahim", "note": 4, "texte": "Très complet, notamment pour les photos."}],
}
CATALOGUE["écouteurs airpods 4"] = {
    "marque": "Apple", "categorie": "Audio", "prix": 125000,
    "description": "Écouteurs sans fil avec audio spatial et boîtier de recharge USB-C.",
    "caracteristiques": {"autonomie": "5 h (30 h avec le boîtier)", "connexion": "Bluetooth", "micro": "oui"},
    "stock": {"blanc": 12},
    "note": 4.5, "nombre_avis": 192,
    "avis": [{"auteur": "Adjoua", "note": 5, "texte": "Connexion simple et son agréable."},
             {"auteur": "Cheick", "note": 4, "texte": "Très pratiques dans les transports."}],
}
CATALOGUE["montre connectée xiaomi"] = {
    "marque": "Xiaomi", "categorie": "Montres", "prix": 32000, "prix_barre": 38000,
    "description": "Suivi d'activité, fréquence cardiaque, sommeil et notifications. Compatible Android et iPhone.",
    "caracteristiques": {"écran": "AMOLED", "autonomie": "14 jours", "étanchéité": "5 ATM"},
    "stock": {"noir": 9, "argent": 4},
    "note": 4.2, "nombre_avis": 231,
    "avis": [{"auteur": "Eric", "note": 5, "texte": "Bonne autonomie et suivi sportif pratique."},
             {"auteur": "Nadège", "note": 3, "texte": "Bon rapport qualité-prix, application perfectible."}],
}

In [5]:
# Rayon MODE : le stock est donné par couleur PUIS par taille
CATALOGUE["baskets nike air force 1"] = {
    "marque": "Nike", "categorie": "Chaussures", "prix": 75000, "prix_barre": 82000,
    "description": "Baskets emblématiques en cuir, semelle épaisse, design intemporel. Unisexe.",
    "caracteristiques": {"matière": "cuir", "style": "lifestyle", "entretien": "chiffon humide"},
    "stock": {"blanc": {"39": 2, "40": 4, "41": 5, "42": 8, "43": 3, "44": 2, "45": 0},
              "noir":  {"39": 1, "40": 3, "41": 2, "42": 0, "43": 5, "44": 2, "45": 1}},
    "note": 4.6, "nombre_avis": 512,
    "avis": [{"auteur": "Franck", "note": 5, "texte": "Très belles baskets, faciles à porter."},
             {"auteur": "Désirée", "note": 4, "texte": "Modèle classique, mais un peu lourd."}],
}
CATALOGUE["hoodie adidas essentials"] = {
    "marque": "Adidas", "categorie": "Vêtements", "prix": 38000, "prix_barre": 45000,
    "description": "Sweat à capuche confortable avec poche kangourou et coupe décontractée.",
    "caracteristiques": {"matière": "coton et polyester", "coupe": "décontractée", "lavage": "30 °C"},
    "stock": {"noir": {"S": 3, "M": 5, "L": 4, "XL": 2}, "gris": {"S": 2, "M": 4, "L": 3, "XL": 1},
              "beige": {"S": 1, "M": 0, "L": 2, "XL": 2}},
    "note": 4.4, "nombre_avis": 146,
    "avis": [{"auteur": "Jean-Marc", "note": 5, "texte": "Très confortable et agréable à porter."},
             {"auteur": "Clarisse", "note": 3, "texte": "Belle qualité, mais taille un peu grand."}],
}
CATALOGUE["jean levis 501"] = {
    "marque": "Levi's", "categorie": "Vêtements", "prix": 55000, "prix_barre": 62000,
    "description": "Jean droit iconique à boutons, en denim résistant. Un classique du quotidien.",
    "caracteristiques": {"coupe": "droite", "matière": "denim", "lavage": "basse température"},
    "stock": {"bleu": {"36": 2, "38": 4, "40": 5, "42": 3, "44": 1}, "noir": {"36": 1, "38": 3, "40": 0, "42": 2, "44": 2}},
    "note": 4.3, "nombre_avis": 278,
    "avis": [{"auteur": "Stéphane", "note": 5, "texte": "Une coupe classique qui dure dans le temps."},
             {"auteur": "Rokia", "note": 4, "texte": "Très joli, mais vérifiez bien la taille."}],
}

In [6]:
# Rayon QUOTIDIEN
CATALOGUE["sac à dos eastpak"] = {
    "marque": "Eastpak", "categorie": "Sacs", "prix": 35000,
    "description": "Sac à dos compact de 24 litres, poche avant zippée et dos rembourré. Pour les cours ou le travail.",
    "caracteristiques": {"capacité": "24 L", "compartiment ordinateur": "non", "garantie": "30 ans"},
    "stock": {"noir": 10, "bleu": 4, "vert": 2},
    "note": 4.7, "nombre_avis": 346,
    "avis": [{"auteur": "Brice", "note": 5, "texte": "Solide, léger et parfait pour les cours."},
             {"auteur": "Estelle", "note": 4, "texte": "Très pratique, mais pas de compartiment ordinateur."}],
}
CATALOGUE["gobelet stanley quencher"] = {
    "marque": "Stanley", "categorie": "Maison", "prix": 28000,
    "description": "Gobelet isotherme de 1,18 L avec poignée, paille et couvercle. Garde le froid jusqu'à 11 h.",
    "caracteristiques": {"capacité": "1,18 L", "matière": "acier inoxydable", "lave-vaisselle": "oui"},
    "stock": {"rose": 3, "crème": 5, "noir": 7},
    "note": 4.5, "nombre_avis": 183,
    "avis": [{"auteur": "Sandrine", "note": 5, "texte": "Grande capacité et poignée très pratique."},
             {"auteur": "Ismaël", "note": 4, "texte": "Joli produit, mais assez encombrant."}],
}

Voici la **vitrine** de Djassa. La mention PROMO signale les produits en promotion.

In [7]:
print(f"{BOUTIQUE['nom']} : {BOUTIQUE['slogan']}\n")
for nom, p in CATALOGUE.items():
    promo = "PROMO" if p.get("prix_barre") else "     "
    print(f"{promo} {p['categorie']:<10} │ {nom:<25} │ {fcfa(p['prix']):>12} │ note {p['note']} ({p['nombre_avis']} avis)")

Djassa : Les bons plans du quotidien, livrés partout en Côte d'Ivoire

PROMO Téléphones │ iphone 16                 │ 690 000 FCFA │ note 4.7 (328 avis)
PROMO Téléphones │ samsung galaxy a36        │ 245 000 FCFA │ note 4.5 (214 avis)
      Audio      │ écouteurs airpods 4       │ 125 000 FCFA │ note 4.5 (192 avis)
PROMO Montres    │ montre connectée xiaomi   │  32 000 FCFA │ note 4.2 (231 avis)
PROMO Chaussures │ baskets nike air force 1  │  75 000 FCFA │ note 4.6 (512 avis)
PROMO Vêtements  │ hoodie adidas essentials  │  38 000 FCFA │ note 4.4 (146 avis)
PROMO Vêtements  │ jean levis 501            │  55 000 FCFA │ note 4.3 (278 avis)
      Sacs       │ sac à dos eastpak         │  35 000 FCFA │ note 4.7 (346 avis)
      Maison     │ gobelet stanley quencher  │  28 000 FCFA │ note 4.5 (183 avis)


**À retenir**
- un agent n'est utile que s'il a accès à **tes** données (stock, prix, avis). Le LLM, lui, ne les connaît pas.

---
## Étape 2 · Pourquoi un LLM seul ne suffit pas

**Objectif** : constater le problème qu'on va résoudre.

On pose une question de client au LLM, **sans lui donner accès au catalogue**.

In [8]:
question = "Bonjour, chez Djassa, l'iPhone 16 rose est en stock ? Il coûte combien ?"

reponse = client.chat.completions.create(
    model=MODELE,
    messages=[{"role": "user", "content": question}],
    **REGLAGES,
)
afficher("reponse", reponse.choices[0].message.content)


Répond       │ Je ne peux pas consulter en temps réel le stock ni les prix d’un magasin. Pour vous aider au mieux, dites-moi :
- de quel magasin Djassa/par quelle ville vous parlez (ou l’URL de la boutique en ligne) ;
- quel modèle exact et quelle capacité vous voulez (iPhone 16, iPhone 16 Plus, 16 Pro… 128/256/512 Go) — vous avez bien dit « rose » ?

Sinon, voici comment vérifier très vite vous‑même et un modèle de message à envoyer au magasin.

Comment vérifier (site / mobile)
1. Allez sur le site de Djassa ou sur l’appli.
2. Tapez « iPhone 16 rose » dans la barre de recherche.
3. Filtrez par modèle et capacité si besoin.
4. Regardez la disponibilité (en ligne / retrait en magasin) et le prix affiché.
5. Si l’info n’apparaît pas, appelez le magasin (numéro sur la page « contact ») ou utilisez le chat en ligne.

Modèle de message / SMS à envoyer au magasin
Bonjour, je voudrais savoir si vous avez en stock un iPhone 16 (rose) en [capacité souhaitée]. Pouvez‑vous me confirmer le prix e

**Que s'est-il passé ?** Le LLM **ne peut pas savoir** : il n'a jamais vu le stock de Djassa, et il ignore que l'iPhone rose est en rupture. Soit il **invente** un prix ou une disponibilité, ce qui est dangereux pour un commerce, soit il répond qu'il ne sait pas, ce qui n'aide pas le client.

**L'idée** : donner au LLM des **outils**, c'est-à-dire des fonctions Python qu'il peut **demander** d'appeler. C'est comme un vendeur qui va vérifier dans la réserve avant de répondre.

<p align="center"><img src="assets/diagrams/nb2_llm_seul_vs_agent.png" alt="Un LLM seul invente sa réponse, alors qu'un agent consulte le catalogue grâce à ses outils" width="900"></p>

**À retenir**
- un LLM connaît le langage, pas ta boutique.

---
## Étape 3 · Créer des outils

**Objectif** : écrire les fonctions qu'Awa pourra utiliser.

Un outil, c'est **une fonction Python normale** qui **renvoie du texte** avec `return`, et non `print`. C'est ce texte que le LLM lira.

On va créer 5 outils, un par type de question client :

| Outil | Question du client |
|---|---|
| `rechercher_produits` | « Vous avez quoi comme téléphones ? », « Un cadeau à moins de 40 000 FCFA ? » |
| `fiche_produit` | « Il vaut le coup ? Que disent les clients ? » |
| `verifier_stock` | « Vous l'avez en noir, en taille 42 ? » |
| `delai_livraison` | « Je le reçois quand ? » |
| `infos_pratiques` | « Je peux le renvoyer ? Je peux payer avec Orange Money ? » |

D'abord, deux petites fonctions pour **tolérer les approximations**. Le LLM écrira parfois « Air Force » au lieu de « baskets nike air force 1 », ou « bleue » au lieu de « bleu ».

In [9]:
import unicodedata

def normaliser(texte):
    """'  Écharpe ' -> 'echarpe' : minuscules, sans accents ni apostrophes."""
    texte = unicodedata.normalize("NFD", texte.lower()).encode("ascii", "ignore").decode()
    return texte.replace("'", " ").split()

def trouver(recherche, choix, strict=False):
    """Renvoie l'élément de `choix` qui partage le plus de mots avec `recherche`, ou None.
    strict=True : tous les mots demandés doivent correspondre (on ne devine pas)."""
    mots = [m for m in normaliser(recherche) if len(m) > 2]
    def score(c):        # « bleue » correspond à « bleu », « telephones » à « telephone »...
        return sum(any(m.startswith(k) or k.startswith(m) for k in normaliser(c) if len(k) > 2) for m in mots)
    meilleur = max(choix, key=score, default=None)
    if meilleur is None or score(meilleur) == 0 or (strict and score(meilleur) < len(mots)):
        return None
    return meilleur

print(trouver("Air Force", CATALOGUE), "|", trouver("Levi's", CATALOGUE), "|", trouver("bleue", ["noir", "bleu"]))

baskets nike air force 1 | jean levis 501 | bleu


**Attention** : Être tolérant a un prix : le client demande des **Nike Air Max**, que Djassa ne vend pas...

In [10]:
print("Tolérant :", trouver("Nike Air Max", CATALOGUE))               # choisit un autre produit
print("Strict   :", trouver("Nike Air Max", CATALOGUE, strict=True))  # ne devine pas

Tolérant : baskets nike air force 1
Strict   : None


Le mode tolérant répond « Air Force 1 » : Awa risquerait d'annoncer un stock pour le **mauvais produit**.
Pour les produits, on utilise donc `strict=True` : en cas de doute, l'outil le dit, et Awa **demande une précision** au lieu de choisir au hasard.

### Les outils « catalogue »

In [11]:
INCONNU = f"Produit introuvable ou ambigu : demande au client de préciser. Produits vendus : {', '.join(CATALOGUE)}."

def rechercher_produits(recherche: str = "", budget_max: int = 0) -> str:
    """Cherche des produits par mot-clé, marque ou catégorie (ex. téléphone, chaussures, promo), avec un budget maximum en FCFA (0 = sans limite)."""
    dans_budget = {n: p for n, p in CATALOGUE.items() if not budget_max or p["prix"] <= budget_max}
    trouves = {n: p for n, p in dans_budget.items()
               if trouver(recherche, [f"{n} {p['marque']} {p['categorie']} {'promo' if p.get('prix_barre') else ''}"])}
    entete = "" if trouves or not recherche else f"Rien ne correspond à « {recherche} ». Produits dans le budget :\n"
    lignes = []
    for nom, p in sorted((trouves or dans_budget).items(), key=lambda x: -x[1]["note"]):   # mieux notés d'abord
        promo = f" au lieu de {fcfa(p['prix_barre'])}" if p.get("prix_barre") else ""
        lignes.append(f"- {nom} ({p['marque']}) : {fcfa(p['prix'])}{promo}, note {p['note']}/5 ({p['nombre_avis']} avis)")
    return entete + ("\n".join(lignes) or "Aucun produit dans ce budget.")

print(rechercher_produits("téléphone"))
print("\n")
print(rechercher_produits("le mieux noté", budget_max=40000))

- iphone 16 (Apple) : 690 000 FCFA au lieu de 730 000 FCFA, note 4.7/5 (328 avis)
- samsung galaxy a36 (Samsung) : 245 000 FCFA au lieu de 265 000 FCFA, note 4.5/5 (214 avis)


Rien ne correspond à « le mieux noté ». Produits dans le budget :
- sac à dos eastpak (Eastpak) : 35 000 FCFA, note 4.7/5 (346 avis)
- gobelet stanley quencher (Stanley) : 28 000 FCFA, note 4.5/5 (183 avis)
- hoodie adidas essentials (Adidas) : 38 000 FCFA au lieu de 45 000 FCFA, note 4.4/5 (146 avis)
- montre connectée xiaomi (Xiaomi) : 32 000 FCFA au lieu de 38 000 FCFA, note 4.2/5 (231 avis)


In [12]:
def fiche_produit(produit: str) -> str:
    """Donne la fiche complète d'un produit : prix, description, caractéristiques, couleurs en stock, note et avis clients."""
    nom = trouver(produit, CATALOGUE, strict=True)
    if nom is None:
        return INCONNU
    p = CATALOGUE[nom]
    prix = fcfa(p['prix']) + (f" au lieu de {fcfa(p['prix_barre'])}" if p.get("prix_barre") else "")
    en_stock = [c for c, q in p["stock"].items() if (sum(q.values()) if isinstance(q, dict) else q) > 0]
    lignes = [f"{nom.upper()} ({p['marque']}) : {prix}", p["description"],
              "Caractéristiques : " + ", ".join(f"{k} : {v}" for k, v in p["caracteristiques"].items()),
              f"Couleurs en stock : {', '.join(en_stock) or 'aucune'}. Note : {p['note']}/5 ({p['nombre_avis']} avis)."]
    lignes += [f"  « {a['texte']} » ({a['auteur']}, {a['note']}/5)" for a in p["avis"]]
    return "\n".join(lignes)

print(fiche_produit("Samsung"))

SAMSUNG GALAXY A36 (Samsung) : 245 000 FCFA au lieu de 265 000 FCFA
Écran AMOLED 6,7 pouces, triple appareil photo et batterie qui tient la journée. Un très bon rapport qualité-prix.
Caractéristiques : stockage : 128 Go, écran : 6,7 pouces, photo : 50 MP, garantie : 2 ans
Couleurs en stock : noir, bleu, argent. Note : 4.5/5 (214 avis).
  « Écran magnifique et téléphone très fluide. » (Salimata, 5/5)
  « Très complet, notamment pour les photos. » (Ibrahim, 4/5)


`verifier_stock` est un peu plus longue, car elle gère **deux cas** : les produits avec une simple couleur (un iPhone noir) et les produits avec couleur **et** taille (des baskets blanches en 42).

In [13]:
def verifier_stock(produit: str, couleur: str, taille: str = "") -> str:
    """Vérifie le stock d'un produit dans une couleur, et dans une taille pour les vêtements et chaussures."""
    nom = trouver(produit, CATALOGUE, strict=True)
    if nom is None:
        return INCONNU
    stock = CATALOGUE[nom]["stock"]
    teinte = trouver(couleur, stock)
    if teinte is None:
        return f"{nom} n'existe pas en {couleur}. Couleurs proposées : {', '.join(stock)}."
    if isinstance(stock[teinte], dict):                         # cas 2 : article avec tailles
        quantite = stock[teinte].get(taille.upper(), 0)
        dispo = [t for t, q in stock[teinte].items() if q > 0]
        if quantite == 0:
            return f"{nom} {teinte} : taille '{taille}' indisponible. Tailles en stock : {', '.join(dispo) or 'aucune'}."
        return f"{nom} {teinte} taille {taille.upper()} : {quantite} en stock."
    if stock[teinte] > 0:                                       # cas 1 : couleur seule
        return f"{nom} {teinte} : {stock[teinte]} en stock."
    autres = [c for c, q in stock.items() if q > 0]
    return f"{nom} {teinte} : rupture de stock. Couleurs disponibles : {', '.join(autres) or 'aucune'}."

print(verifier_stock("iPhone 16", "rose"))
print(verifier_stock("Air Force", "blanches", "42"))
print(verifier_stock("hoodie", "beige", "M"))

iphone 16 rose : rupture de stock. Couleurs disponibles : noir, blanc, turquoise.
baskets nike air force 1 blanc taille 42 : 8 en stock.
hoodie adidas essentials beige : taille 'M' indisponible. Tailles en stock : S, L, XL.


### Les outils « service client »

In [14]:
JOURS = ["lundi", "mardi", "mercredi", "jeudi", "vendredi", "samedi", "dimanche"]

def dans_jours_ouvres(n):
    """Date dans n jours ouvrés (on saute samedi et dimanche), ex. 'jeudi 29/10'."""
    jour = AUJOURDHUI
    while n > 0:
        jour += timedelta(days=1)
        if jour.weekday() < 5:
            n -= 1
    return f"{JOURS[jour.weekday()]} {jour:%d/%m}"

def cle(texte):
    """'  Port-Bouët ' -> 'port bouet' : pour comparer sans se soucier des accents ni de la casse."""
    return " ".join(normaliser(texte.replace("-", " ")))

def delai_livraison(ville: str) -> str:
    """Estime les dates de livraison standard et express à domicile, à partir de la commune d'Abidjan du client (ex. Cocody) ou de sa ville (Bouaké, Yamoussoukro, San Pedro)."""
    zone = "Abidjan" if cle(ville) in [cle(v) for v in ZONES["Abidjan"]["villes"]] else "Intérieur"
    (jours_std, frais_std), (jours_exp, frais_exp) = ZONES[zone]["standard"], ZONES[zone]["express"]
    return (f"Standard : {dans_jours_ouvres(jours_std)} (offerte dès {fcfa(BOUTIQUE['livraison_gratuite_des'])}, sinon {fcfa(frais_std)}). "
            f"Express : {dans_jours_ouvres(jours_exp)} ({fcfa(frais_exp)}).")

In [15]:
def infos_pratiques() -> str:
    """Donne les conditions de retour et de remboursement, les moyens de paiement et les horaires du service client."""
    return (f"Retours : {POLITIQUE_RETOUR} Paiement : {', '.join(MOYENS_PAIEMENT)}. "
            f"Service client : {BOUTIQUE['service_client']}.")

print(delai_livraison("Cocody"))
print(infos_pratiques())

Standard : jeudi 29/10 (offerte dès 50 000 FCFA, sinon 1 500 FCFA). Express : mercredi 28/10 (2 500 FCFA).
Retours : Retour sous 14 jours, article non utilisé et dans son emballage d'origine (gratuit en point relais Djassa). Remboursement sous 5 jours ouvrés après réception du colis, sur le moyen de paiement utilisé. Paiement : Orange Money, MTN MoMo, Wave, carte bancaire, paiement à la livraison (Abidjan uniquement), paiement en 3 fois dès 150 000 FCFA. Service client : du lundi au samedi, de 8 h à 18 h, au +225 07 00 00 00 00 (appel ou WhatsApp).


### En Python pur : décrire l'outil au LLM

Le LLM ne lit pas ton code Python. Il faut lui envoyer une **fiche descriptive** de chaque outil au format JSON : son nom, à quoi il sert et ses paramètres.

<p align="center"><img src="assets/diagrams/nb2_anatomie_outil.png" alt="Une fonction Python (nom, paramètres, docstring, return) est convertie en fiche JSON, la seule chose que le LLM voit" width="860"></p>

**Attention** : Cette description est **un prompt** : plus elle est claire, mieux le LLM choisira le bon outil. Voici la fiche de `verifier_stock`, écrite à la main :

In [16]:
schema_stock = {
    "type": "function",
    "function": {
        "name": "verifier_stock",
        "description": "Vérifie le stock d'un produit dans une couleur, et dans une taille pour les vêtements et chaussures.",
        "parameters": {
            "type": "object",
            "properties": {
                "produit": {"type": "string", "description": "Nom du produit, ex. iphone 16"},
                "couleur": {"type": "string", "description": "Couleur demandée, ex. noir"},
                "taille":  {"type": "string", "description": "Taille, ex. 42 ou M (vide si sans taille)"},
            },
            "required": ["produit", "couleur"],          # la taille est facultative
        },
    },
}

Écrire 5 fiches à la main serait long. La fonction `decrire()` les **fabrique automatiquement** en lisant la docstring et la signature de chaque fonction : le nom des paramètres, leur type et s'ils sont obligatoires.

In [17]:
import inspect
TYPES_JSON = {str: "string", float: "number", int: "integer", bool: "boolean"}

def decrire(fonction):
    """Fabrique la fiche JSON d'un outil à partir de sa docstring et de sa signature."""
    params = inspect.signature(fonction).parameters
    proprietes = {nom: {"type": TYPES_JSON[p.annotation]} for nom, p in params.items()}
    obligatoires = [nom for nom, p in params.items() if p.default is inspect.Parameter.empty]
    return {"type": "function", "function": {"name": fonction.__name__, "description": fonction.__doc__,
            "parameters": {"type": "object", "properties": proprietes, "required": obligatoires}}}

FONCTIONS = [rechercher_produits, fiche_produit, verifier_stock, delai_livraison, infos_pratiques]
SCHEMAS = [decrire(f) for f in FONCTIONS]
OUTILS = {f.__name__: f for f in FONCTIONS}            # nom -> fonction, pour les exécuter
print(json.dumps(SCHEMAS[3], ensure_ascii=False, indent=1))

{
 "type": "function",
 "function": {
  "name": "delai_livraison",
  "description": "Estime les dates de livraison standard et express à domicile, à partir de la commune d'Abidjan du client (ex. Cocody) ou de sa ville (Bouaké, Yamoussoukro, San Pedro).",
  "parameters": {
   "type": "object",
   "properties": {
    "ville": {
     "type": "string"
    }
   },
   "required": [
    "ville"
   ]
  }
 }
}


**À savoir** : `decrire()` est un **générateur pédagogique** : il ne gère que des paramètres simples, sans exemples ni valeurs autorisées. En production, on écrit un schéma explicite, souvent avec **Pydantic** (étape 6).

### Avec LangChain

LangChain fait exactement ce que fait `decrire()` : il lit **le nom, la docstring et les annotations de type** de ta fonction.

In [18]:
from langchain_core.tools import tool

outils_lc = [tool(f) for f in FONCTIONS]

for o in outils_lc:
    print(f"{o.name:<20} │ paramètres : {list(o.args)}")

rechercher_produits  │ paramètres : ['recherche', 'budget_max']
fiche_produit        │ paramètres : ['produit']
verifier_stock       │ paramètres : ['produit', 'couleur', 'taille']
delai_livraison      │ paramètres : ['ville']
infos_pratiques      │ paramètres : []


**À retenir**
- Un outil = **une fonction Python qui renvoie du texte** + **une description** pour le LLM.
- En Python pur, tu écris la description en JSON. LangChain la génère à partir de la docstring.
- Une bonne docstring, c'est un bon outil.

---
## Étape 4 · Le tool calling

**Objectif** : voir comment le LLM **demande** à utiliser un outil.

Le point clé : **le LLM n'exécute jamais rien lui-même**. Il répond : *« Appelle `verifier_stock` avec `produit=air force`, `couleur=blanc` et `taille=42` »*, et c'est **ton code** qui exécute la fonction.

<p align="center"><img src="assets/diagrams/nb2_tool_calling.png" alt="Séquence du tool calling : le code envoie la question, le LLM demande un outil, le code l'exécute et renvoie le résultat" width="760"></p>

On commence par le **prompt système**, la « fiche de poste » d'Awa. Il indique **quel outil utiliser pour quelle question**, ce qui aide beaucoup les petits modèles locaux.

In [19]:
SYSTEME = f"""Tu es Awa, conseillère de vente de la boutique en ligne {BOUTIQUE['nom']} ({BOUTIQUE['slogan']}).
Nous sommes le {JOURS[AUJOURDHUI.weekday()]} {AUJOURDHUI:%d/%m/%Y}. Les prix sont en FCFA.
N'invente jamais : chaque information doit venir d'un outil.
Tu consultes seulement : tu ne peux ni réserver, ni commander, ni rembourser.
- Disponibilité d'une couleur ou d'une taille : verifier_stock.
- Date de livraison : delai_livraison. Retours, paiement, horaires : infos_pratiques.
- Conseil ou recherche de produits : rechercher_produits, puis fiche_produit pour les avis.
Si le client pose plusieurs questions, appelle tous les outils nécessaires avant de répondre.
Ne dis jamais « je vais vérifier » : vérifie tout de suite avec l'outil.
Si une information du client manque ou est invalide (commune, taille...), demande-la : ne la devine jamais.
Réponds en français, en 4 phrases maximum, avec un ton chaleureux."""
print(SYSTEME)

Tu es Awa, conseillère de vente de la boutique en ligne Djassa (Les bons plans du quotidien, livrés partout en Côte d'Ivoire).
Nous sommes le mardi 27/10/2026. Les prix sont en FCFA.
N'invente jamais : chaque information doit venir d'un outil.
Tu consultes seulement : tu ne peux ni réserver, ni commander, ni rembourser.
- Disponibilité d'une couleur ou d'une taille : verifier_stock.
- Date de livraison : delai_livraison. Retours, paiement, horaires : infos_pratiques.
- Conseil ou recherche de produits : rechercher_produits, puis fiche_produit pour les avis.
Si le client pose plusieurs questions, appelle tous les outils nécessaires avant de répondre.
Ne dis jamais « je vais vérifier » : vérifie tout de suite avec l'outil.
Si une information du client manque ou est invalide (commune, taille...), demande-la : ne la devine jamais.
Réponds en français, en 4 phrases maximum, avec un ton chaleureux.


### 4a · Le LLM demande un outil

In [20]:
messages = [{"role": "system", "content": SYSTEME},
            {"role": "user", "content": "Vous avez les Nike Air Force 1 blanches en 42 ?"}]

reponse = client.chat.completions.create(model=MODELE, messages=messages, tools=SCHEMAS, **REGLAGES)
demande = reponse.choices[0].message

print("Texte du LLM :", repr(demande.content))          # souvent vide : il préfère demander un outil
for appel in demande.tool_calls or []:
    afficher("outil", f"{appel.function.name}({appel.function.arguments})")

Texte du LLM : None
Appelle      │ verifier_stock({"produit":"Nike Air Force 1","couleur":"blanc","taille":"42"})


Le LLM n'a pas répondu au client : il a renvoyé **une demande d'appel**, avec le nom de l'outil et ses arguments en JSON.

### 4b · Ton code exécute l'outil et renvoie le résultat

In [21]:
def executer(nom, arguments):
    """Exécute l'outil demandé par le LLM (on la renforcera à l'étape 6)."""
    return OUTILS[nom](**arguments)

appel = demande.tool_calls[0]
arguments = json.loads(appel.function.arguments)        # texte JSON -> dictionnaire Python
resultat = executer(appel.function.name, arguments)
afficher("resultat", resultat)

messages.append(demande)                                # on garde la demande du LLM...
messages.append({"role": "tool", "tool_call_id": appel.id, "content": resultat})  # ...et le résultat

final = client.chat.completions.create(model=MODELE, messages=messages, tools=SCHEMAS, **REGLAGES)
afficher("reponse", final.choices[0].message.content)

Reçoit       │ baskets nike air force 1 blanc taille 42 : 8 en stock.



Répond       │ Bonne nouvelle — nous avons 8 paires de Nike Air Force 1 blanches en taille 42 en stock. Voulez-vous que je vérifie le prix, d'autres tailles/couleurs ou le délai de livraison pour votre commune ? Je peux aussi vous indiquer comment commander si vous le souhaitez.


### Avec LangChain

`bind_tools()` « attache » les outils au LLM. Les arguments arrivent déjà convertis en dictionnaire Python.

In [22]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=MODELE, base_url=BASE_URL, api_key=CLE, **REGLAGES)   # marche aussi avec Ollama
llm_avec_outils = llm.bind_tools(outils_lc)

demande_lc = llm_avec_outils.invoke([("system", SYSTEME), ("user", "Vous avez les Nike Air Force 1 blanches en 42 ?")])
for appel in demande_lc.tool_calls:
    afficher("outil", f"{appel['name']}({appel['args']})")

Appelle      │ verifier_stock({'produit': 'Nike Air Force 1', 'couleur': 'blanc', 'taille': '42'})


**À retenir**
- Le LLM **choisit** l'outil et ses arguments, mais c'est **ton code** qui l'exécute. C'est toi qui gardes le contrôle.
- Le résultat est renvoyé au LLM dans un message de rôle `"tool"`, puis le LLM rédige la réponse au client.
- **À savoir** : Pour un nouveau projet, OpenAI recommande son API *Responses*. On garde ici *Chat Completions*, dont la boucle est très lisible.

---
## Étape 5 · La boucle ReAct : de l'outil à l'agent

**Objectif** : laisser le LLM enchaîner **plusieurs outils** tout seul, jusqu'à avoir de quoi répondre.

À l'étape 4, on a fait **un seul aller-retour à la main**. Mais pour conseiller un cadeau, Awa doit d'abord **chercher** des produits, puis **lire les avis**, puis **répondre**. **ReAct** (*Reason + Act*) consiste à répéter le cycle suivant :

<p align="center"><img src="assets/diagrams/nb2_boucle_react.png" alt="Boucle ReAct : réfléchir, agir, observer, puis répondre quand le LLM n'a plus besoin d'outil" width="860"></p>

**Attention** : La « réflexion » se passe **dans** le modèle : on ne la voit pas. Ce qu'on peut observer et vérifier, ce sont ses **décisions** : quel outil, quels arguments, quel résultat. Si le LLM écrit un petit texte en plus, on l'affiche avec l'étiquette Note, sans le prendre pour son vrai raisonnement.

C'est **le LLM qui décide** quand s'arrêter. C'est ce qui fait la différence entre un simple script et un **agent**.

### En Python pur : une simple boucle `for`

In [23]:
def roles(messages):
    """['system', 'user', 'assistant (demande d'outil)', 'tool'] : les rôles des messages, dans l'ordre."""
    noms = [m["role"] if isinstance(m, dict) else m.role for m in messages]
    return [n + " (demande d'outil)" if not isinstance(m, dict) and m.tool_calls else n for n, m in zip(noms, messages)]

def agent_python(question, max_tours=6, montrer_etat=False):
    messages = [{"role": "system", "content": SYSTEME}, {"role": "user", "content": question}]
    for tour in range(1, max_tours + 1):             # garde-fou : jamais de boucle infinie
        if montrer_etat:
            print(f"   Tour {tour} : le LLM reçoit {len(messages)} messages → {', '.join(roles(messages))}")
        reponse = client.chat.completions.create(model=MODELE, messages=messages, tools=SCHEMAS, **REGLAGES)
        msg = reponse.choices[0].message
        messages.append(msg)
        if not msg.tool_calls:                       # plus d'outil demandé -> réponse finale
            return afficher("reponse", msg.content)
        if msg.content:
            afficher("pensee", msg.content)          # texte éventuel (pas son raisonnement interne)
        for appel in msg.tool_calls:
            arguments = json.loads(appel.function.arguments or "{}")
            afficher("outil", f"{appel.function.name}({arguments})")        # AGIR
            resultat = executer(appel.function.name, arguments)
            afficher("resultat", resultat)                                  # OBSERVER
            messages.append({"role": "tool", "tool_call_id": appel.id, "content": resultat})
    afficher("reponse", "Désolée, je n'ai pas réussi à répondre. Un conseiller va vous recontacter.")

In [24]:
agent_python("Je cherche un cadeau à moins de 40 000 FCFA. Qu'est-ce qui est le mieux noté chez vous ?")

Appelle      │ rechercher_produits({'recherche': 'cadeau', 'budget_max': 40000})
Reçoit       │ Rien ne correspond à « cadeau ». Produits dans le budget :
- sac à dos eastpak (Eastpak) : 35 000 FCFA, note 4.7/5 (346 avis)
- gobelet stanley quencher (Stanley) : 28 000 FCFA, note 4.5/5 (183 avis)
- hoodie adidas essentials (Adidas) : 38 000 FCFA au lieu de 45 000 FCFA, note 4.4/5 (146 avis)
- montre connectée xiaomi (Xiaomi) : 32 000 FCFA au lieu de 38 000 FCFA, note 4.2/5 (231 avis)



Répond       │ Le meilleur produit sous 40 000 FCFA est le sac à dos Eastpak — 35 000 FCFA, note 4,7/5 (346 avis).  
En alternatives : gobelet Stanley Quencher 28 000 FCFA (4,5/5) et hoodie Adidas en promo 38 000 FCFA (4,4/5).  
Voulez-vous que j’affiche la fiche détaillée du sac à dos ou que je vérifie la disponibilité par couleur/taille pour vous ?


### Ce que le LLM reçoit à chaque tour

La boucle ne fait qu'une chose : **agrandir la liste `messages`** et la renvoyer **en entier** au LLM, exactement comme l'historique du notebook 1. À chaque tour, le LLM ne se souvient de rien : il relit tout. Avec `montrer_etat=True`, on voit cette liste grandir :

In [25]:
agent_python("L'iPhone 16 noir est dispo ? Et je le reçois quand à Cocody ?", montrer_etat=True)

   Tour 1 : le LLM reçoit 2 messages → system, user


Appelle      │ verifier_stock({'produit': 'iPhone 16', 'couleur': 'Noir'})
Reçoit       │ iphone 16 noir : 8 en stock.
Appelle      │ delai_livraison({'ville': 'Cocody'})
Reçoit       │ Standard : jeudi 29/10 (offerte dès 50 000 FCFA, sinon 1 500 FCFA). Express : mercredi 28/10 (2 500 FCFA).
   Tour 2 : le LLM reçoit 5 messages → system, user, assistant (demande d'outil), tool, tool



Répond       │ Bonne nouvelle — l'iPhone 16 noir est en stock : 8 exemplaires disponibles.  
Livraison à Cocody : standard jeudi 29/10 (offerte dès 50 000 FCFA, sinon 1 500 FCFA), ou express mercredi 28/10 (2 500 FCFA).  
Souhaitez-vous que je vous envoie la fiche détaillée (prix, caractéristiques et avis) ?


Lis la trace comme un état qui évolue :

- **Tour 1** : le LLM reçoit **2 messages** (`system`, `user`). Il répond par une seule **demande d'outils** (ici deux outils à la fois).
- Ton code exécute les outils et **ajoute** leurs résultats : un message `tool` par résultat. La liste passe à **5 messages**.
- **Tour 2** : le LLM relit les 5 messages et, n'ayant plus besoin d'outil, **répond au client**. La boucle s'arrête parce que le LLM ne demande plus rien.

Le LLM ne garde donc rien d'un tour à l'autre : tout ce qu'il « sait » à un instant donné est dans cette liste, que **ton code** construit.

### Avec LangGraph

LangGraph sert à construire des agents sous forme de **graphe**. Notre boucle devient deux **nœuds** (des cases) reliés par des **flèches** :

<p align="center"><img src="assets/diagrams/nb2_graphe_langgraph.png" alt="Graphe LangGraph : nœud llm, condition tools_condition, nœud outils, retour au llm, fin" width="880"></p>

| Élément LangGraph | Rôle | Équivalent dans `agent_python` |
|---|---|---|
| `MessagesState` | La mémoire partagée : la liste des messages | la liste `messages` |
| nœud `llm` | Appelle le LLM avec les outils | `client.chat.completions.create(...)` |
| nœud `outils` (`ToolNode`) | Exécute les outils demandés | la boucle `for appel in ...` |
| `tools_condition` | Décide : vers les outils, ou fin | `if not msg.tool_calls` |

In [26]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition

def construire_agent(outils):
    llm_outils = llm.bind_tools(outils)
    def noeud_llm(etat):                              # le LLM lit l'historique et décide
        return {"messages": [llm_outils.invoke([("system", SYSTEME)] + etat["messages"])]}
    graphe = StateGraph(MessagesState)
    graphe.add_node("llm", noeud_llm)
    graphe.add_node("outils", ToolNode(outils))       # exécute les outils demandés
    graphe.add_edge(START, "llm")
    graphe.add_conditional_edges("llm", tools_condition, {"tools": "outils", END: END})
    graphe.add_edge("outils", "llm")                  # après les outils, retour au LLM
    return graphe.compile()

agent_lg = construire_agent(outils_lc)
print("Agent LangGraph prêt :", list(agent_lg.get_graph().nodes))

Agent LangGraph prêt : ['__start__', 'llm', 'outils', '__end__']


Comme pour la version Python, une petite fonction transforme l'historique des messages en traces lisibles.

In [27]:
def afficher_messages(messages):
    """Traduit l'historique LangGraph en traces Note, Appelle, Reçoit et Répond."""
    for m in messages:
        if m.type == "ai" and m.tool_calls:
            if m.content:
                afficher("pensee", m.content)
            for appel in m.tool_calls:
                afficher("outil", f"{appel['name']}({appel['args']})")
        elif m.type == "tool":
            afficher("resultat", m.content)
        elif m.type == "ai":
            afficher("reponse", m.content)

In [28]:
resultat = agent_lg.invoke({"messages": [("user", "Je cherche un cadeau à moins de 40 000 FCFA. Qu'est-ce qui est le mieux noté chez vous ?")]})
afficher_messages(resultat["messages"])

Appelle      │ rechercher_produits({'recherche': 'cadeau', 'budget_max': 40000})
Reçoit       │ Rien ne correspond à « cadeau ». Produits dans le budget :
- sac à dos eastpak (Eastpak) : 35 000 FCFA, note 4.7/5 (346 avis)
- gobelet stanley quencher (Stanley) : 28 000 FCFA, note 4.5/5 (183 avis)
- hoodie adidas essentials (Adidas) : 38 000 FCFA au lieu de 45 000 FCFA, note 4.4/5 (146 avis)
- montre connectée xiaomi (Xiaomi) : 32 000 FCFA au lieu de 38 000 FCFA, note 4.2/5 (231 avis)
Appelle      │ fiche_produit({'produit': 'sac à dos eastpak'})
Reçoit       │ SAC À DOS EASTPAK (Eastpak) : 35 000 FCFA
Sac à dos compact de 24 litres, poche avant zippée et dos rembourré. Pour les cours ou le travail.
Caractéristiques : capacité : 24 L, compartiment ordinateur : non, garantie : 30 ans
Couleurs en stock : noir, bleu, vert. Note : 4.7/5 (346 avis).
  « Solide, léger et parfait pour les cours. » (Brice, 5/5)
  « Très pratique, mais pas de compartiment ordinateur. » (Estelle, 4/5)

Répond      

**À retenir**
- Un **agent**, c'est un LLM + des outils + **une boucle**. Le LLM décide de l'étape suivante.
- Il faut toujours un **garde-fou** (`max_tours`) pour éviter une boucle sans fin. Dans LangGraph, c'est `recursion_limit`.
- LangGraph rend la boucle **visible** sous forme de graphe. Tu peux ensuite ajouter des nœuds, par exemple une validation humaine avant un remboursement.
- **À savoir** : Raccourci : `create_agent` (dans le package `langchain`) construit ce même graphe en une seule ligne.

---
## Étape 6 · Sécuriser les outils avec Pydantic

**Objectif** : vérifier les arguments envoyés par le LLM **avant** d'exécuter un outil.

Le LLM n'est pas infaillible. Il peut envoyer `ville="Abidjan"` au lieu d'une commune comme `"Cocody"`, ou `taille=42` (un nombre) au lieu de `"42"` (un texte). Entre la demande du LLM et l'exécution, **ton programme** a le dernier mot :

<p align="center"><img src="assets/diagrams/nb2_validation_pydantic.png" alt="Le programme valide le format puis les règles métier avant d'exécuter l'outil, sinon l'erreur revient au LLM" width="900"></p>

Regardons ce qui se passe sans validation :

In [29]:
print(delai_livraison("Korhogo"))                    # une ville qu'on ne dessert pas : pas d'erreur... mais une date inventée
try:
    verifier_stock("air force", "blanc", 42)         # un nombre au lieu d'un texte
except Exception as erreur:
    print("Plantage :", repr(erreur))

Standard : lundi 02/11 (offerte dès 50 000 FCFA, sinon 3 000 FCFA). Express : jeudi 29/10 (5 000 FCFA).
Plantage : AttributeError("'int' object has no attribute 'upper'")


Deux problèmes : une **réponse fausse en silence**, et un **plantage** qui casserait la boucle de l'agent.

**Pydantic** est la bibliothèque Python de référence pour **valider des données**. On décrit la forme attendue des arguments dans une classe, et Pydantic :
- **corrige** ce qui peut l'être, par exemple `42` devient `"42"` ;
- **refuse** ce qui est invalide, avec un message clair, par exemple « cette ville n'est pas desservie ».

In [30]:
from pydantic import BaseModel, ConfigDict, Field, ValidationError, field_validator

VILLES = {cle(v): v for zone in ZONES.values() for v in zone["villes"]}       # les communes et villes desservies

class ArgsLivraison(BaseModel):
    ville: str = Field(description="Commune d'Abidjan (ex. Cocody, Yopougon) ou ville desservie (Bouaké, Yamoussoukro, San Pedro)")

    @field_validator("ville")
    @classmethod
    def ville_desservie(cls, valeur):
        if cle(valeur) not in VILLES:                                           # « Abidjan » seul ne suffit pas : il faut la commune
            raise ValueError("ville non desservie : donne une commune d'Abidjan (ex. Cocody) ou Bouaké, Yamoussoukro, San Pedro")
        return VILLES[cle(valeur)]

class ArgsStock(BaseModel):
    model_config = ConfigDict(coerce_numbers_to_str=True)       # 42 -> "42"
    produit: str = Field(description="Nom du produit, ex. iphone 16")
    couleur: str = Field(description="Couleur demandée, ex. noir")
    taille: str = Field("", description="Taille, ex. 42 ou M (vide si sans taille)")

print("Valide :", ArgsStock(produit="air force", couleur="blanc", taille=42))
try:
    ArgsLivraison(ville="Abidjan")
except ValidationError as erreur:
    print("Refusé :", erreur.errors()[0]["msg"])

Valide : produit='air force' couleur='blanc' taille='42'
Refusé : Value error, ville non desservie : donne une commune d'Abidjan (ex. Cocody) ou Bouaké, Yamoussoukro, San Pedro


### En Python pur : brancher la validation dans l'agent

On remplace `executer()` par une version qui **valide d'abord**. L'astuce : en cas d'erreur, on ne plante pas, on **renvoie l'erreur au LLM** comme résultat de l'outil. Grâce à la boucle ReAct, il peut alors **se corriger** ou **demander l'information au client**.

Comme `agent_python()` appelle `executer()`, il profite tout de suite de la nouvelle version.

In [31]:
VALIDATEURS = {"delai_livraison": ArgsLivraison, "verifier_stock": ArgsStock}

def executer(nom, arguments):
    """Valide les arguments avec Pydantic, puis exécute l'outil. En cas d'erreur, prévient le LLM."""
    if nom not in OUTILS:
        return f"Outil inconnu : {nom}."
    if nom in VALIDATEURS:
        try:
            arguments = VALIDATEURS[nom](**arguments).model_dump()
        except ValidationError as erreur:
            detail = erreur.errors()[0]
            return f"Arguments invalides ({detail['loc'][0]} : {detail['msg']}). Corrige-les ou demande l'information au client."
    try:
        return OUTILS[nom](**arguments)
    except Exception as erreur:                         # panne, délai dépassé...
        return f"Erreur technique ({erreur}). N'invente rien : propose au client de réessayer plus tard."

print(executer("delai_livraison", {"ville": "Abidjan"}))
print(executer("verifier_stock", {"produit": "air force", "couleur": "blanc", "taille": 42}))

Arguments invalides (ville : Value error, ville non desservie : donne une commune d'Abidjan (ex. Cocody) ou Bouaké, Yamoussoukro, San Pedro). Corrige-les ou demande l'information au client.
baskets nike air force 1 blanc taille 42 : 8 en stock.


**À toi de prédire.** Dans la cellule suivante, le client dit seulement « J'habite à Abidjan », sans préciser sa commune. Avant d'exécuter, décide : que va répondre Awa ?

- **(a)** une date de livraison, car elle aura choisi une commune toute seule ;
- **(b)** une demande au client de préciser sa commune.

<details>
<summary>Voir la réponse</summary>

Normalement **(b)** : Pydantic refuse `Abidjan`, l'erreur revient à Awa comme résultat de l'outil, et la consigne « ne la devine jamais » lui dit de redemander. Si tu observes **(a)**, c'est que le LLM a deviné une commune (par exemple Cocody) : c'est exactement le risque décrit plus bas, et la raison de cette consigne dans le prompt.

</details>

In [32]:
agent_python("Je voudrais le sac Eastpak noir, je le reçois quand ? J'habite à Abidjan.")

Appelle      │ verifier_stock({'produit': 'Sac Eastpak', 'couleur': 'noir'})
Reçoit       │ sac à dos eastpak noir : 10 en stock.
Appelle      │ delai_livraison({'ville': 'Abidjan'})
Reçoit       │ Arguments invalides (ville : Value error, ville non desservie : donne une commune d'Abidjan (ex. Cocody) ou Bouaké, Yamoussoukro, San Pedro). Corrige-les ou demande l'information au client.



Répond       │ Bonne nouvelle — le sac Eastpak noir est disponible, il y a 10 unités en stock.  
Pour vous donner la date de livraison à Abidjan, pouvez-vous préciser votre commune (par ex. Cocody) ?  
Dès que vous me donnez la commune, je vous indique les délais standard et express.  
Je peux vérifier les délais et conditions, mais je ne peux pas réserver ni commander pour vous.


Le client n'a pas dit dans quelle **commune** d'Abidjan il habite. Pydantic refuse l'argument `Abidjan` (ce n'est pas une commune livrable), et Awa reçoit l'erreur comme résultat de l'outil. Au lieu d'inventer une date, elle **demande la commune** au client.

### Avec LangChain

On donne le modèle Pydantic à `tool()` avec `args_schema`. LangChain s'en sert pour **fabriquer la fiche JSON** (avec les descriptions des champs) et pour **valider les arguments**. Dans LangGraph, `ToolNode` renvoie automatiquement l'erreur au LLM, comme notre `executer()`.

In [33]:
outils_valides = [tool(rechercher_produits), tool(fiche_produit), tool(infos_pratiques),
                  tool(verifier_stock, args_schema=ArgsStock),
                  tool(delai_livraison, args_schema=ArgsLivraison)]

agent_lg = construire_agent(outils_valides)                      # même graphe, outils sécurisés
afficher_messages(agent_lg.invoke({"messages": [("user", "Je voudrais le sac Eastpak noir, je le reçois quand ? J'habite à Abidjan.")]})["messages"])

Appelle      │ verifier_stock({'produit': 'sac Eastpak', 'couleur': 'noir', 'taille': ''})
Appelle      │ delai_livraison({'ville': 'Abidjan'})
Reçoit       │ sac à dos eastpak noir : 10 en stock.
Reçoit       │ Error invoking tool 'delai_livraison' with kwargs {'ville': 'Abidjan'} with error:
 ville: Value error, ville non desservie : donne une commune d'Abidjan (ex. Cocody) ou Bouaké, Yamoussoukro, San Pedro
 Please fix the error and try again.

Répond       │ Bonne nouvelle — le sac Eastpak noir est en stock (10 exemplaires disponibles). Pour que je vous indique la date de livraison à Abidjan, précisez votre commune (ex. Cocody, Yopougon, Marcory). Je ne peux ni réserver ni commander, mais dès que vous me donnez la commune je vous donne le délai.


### Et quand c'est le système qui casse ?

Un agent rencontre aussi des erreurs qui ne viennent pas des arguments :

| Problème | Ce que fait le programme |
|---|---|
| Outil inconnu | répond « outil inconnu » au LLM |
| Arguments JSON illisibles | renvoie l'erreur au LLM, ou s'arrête proprement |
| Catalogue indisponible, délai dépassé | renvoie « erreur technique », sans planter |
| Résultat vide ou incohérent | contrôle métier avant de répondre |

Simulons une **panne du catalogue** :

In [34]:
def stock_en_panne(**arguments):
    raise TimeoutError("le catalogue ne répond pas")

OUTILS["verifier_stock"] = stock_en_panne        # on simule la panne
agent_python("Vous avez l'iPhone 16 en noir ?")
OUTILS["verifier_stock"] = verifier_stock        # on répare

Appelle      │ verifier_stock({'produit': 'iPhone 16', 'couleur': 'noir'})
Reçoit       │ Erreur technique (le catalogue ne répond pas). N'invente rien : propose au client de réessayer plus tard.



Répond       │ Désolée, le catalogue rencontre une erreur technique et je ne peux pas vérifier la disponibilité de l'iPhone 16 en noir pour l'instant. Pouvez-vous réessayer dans quelques minutes ? Si vous préférez, je peux chercher d'autres modèles ou couleurs disponibles à la place. Dites-moi votre préférence et je m'en occupe.


Awa n'invente pas de stock et ne réessaie pas sans fin (`max_tours` l'en empêche). **Une boucle ReAct ne remplace pas les garde-fous d'un logiciel.**

**À retenir**
- Les arguments venant d'un LLM sont des **données non fiables**, comme un formulaire rempli par un inconnu. **On les valide toujours.**
- On **renvoie l'erreur au LLM** au lieu de planter : la boucle ReAct lui permet de se corriger.
- **Attention** : « se corriger » ne doit pas vouloir dire **deviner** : sans consigne, un LLM peut choisir une commune à la place du client (« Cocody » au lieu d'« Abidjan »). C'est pourquoi le prompt système dit : *« demande-la : ne la devine jamais »*.
- Bonus : la classe Pydantic sert **à la fois** à valider et à décrire l'outil au LLM.

Pydantic ne protège qu'**un** des trois niveaux :

| Niveau | Exemple | Protection |
|---|---|---|
| Format | `Abidjan` au lieu d'une commune desservie | Pydantic |
| Métier | taille 46 inexistante pour ce produit | l'outil, les règles métier |
| Autorisation | réserver ou rembourser | règles de l'application + validation humaine |

`42 → "42"` est une conversion pratique, mais elle ne dit pas si la taille existe, ni si la demande est légitime.

---
## Étape 7 · Le scénario complet

**Objectif** : un vrai message client, avec **plusieurs questions en une**.

Observe les traces : Awa doit **décider seule** quels outils utiliser. Le piège : l'iPhone rose est en rupture, et la livraison standard vers Cocody n'arrive pas avant jeudi. Saura-t-elle proposer une autre couleur et l'option express ?

In [35]:
CLIENT = ("Bonjour ! L'iPhone 16 rose est dispo ? Sinon, quelles couleurs vous reste-t-il ? "
          "J'habite à Cocody : je peux l'avoir avant jeudi ? Et je peux le renvoyer s'il ne me plaît pas ?")

print("════ Version Python pur ════")
agent_python(CLIENT)

════ Version Python pur ════


Appelle      │ verifier_stock({'produit': 'iPhone 16', 'couleur': 'rose'})
Reçoit       │ iphone 16 rose : rupture de stock. Couleurs disponibles : noir, blanc, turquoise.
Appelle      │ fiche_produit({'produit': 'iPhone 16'})
Reçoit       │ IPHONE 16 (Apple) : 690 000 FCFA au lieu de 730 000 FCFA
Écran 6,1 pouces, puce A18, appareil photo 48 MP et USB-C. Idéal pour la photo et les réseaux sociaux.
Caractéristiques : stockage : 128 Go, écran : 6,1 pouces, photo : 48 MP, garantie : 2 ans
Couleurs en stock : noir, blanc, turquoise. Note : 4.7/5 (328 avis).
  « Très bon téléphone, rapide et excellent en photo. » (Fatou, 5/5)
  « Très agréable à utiliser, mais le prix reste élevé. » (Kouamé, 4/5)
Appelle      │ delai_livraison({'ville': 'Cocody'})
Reçoit       │ Standard : jeudi 29/10 (offerte dès 50 000 FCFA, sinon 1 500 FCFA). Express : mercredi 28/10 (2 500 FCFA).
Appelle      │ infos_pratiques({})
Reçoit       │ Retours : Retour sous 14 jours, article non utilisé et dans son emballage 


Répond       │ Bonjour ! Le iPhone 16 (690 000 FCFA) rose est en rupture de stock ; il est disponible en noir, blanc et turquoise.  
Depuis Cocody, la livraison standard arrive le jeudi 29/10 (offerte dès 50 000 FCFA sinon 1 500 FCFA) et l'option express permet d'avoir le téléphone avant jeudi, le mercredi 28/10 (2 500 FCFA).  
Vous pouvez le renvoyer sous 14 jours si l'article est non utilisé et dans son emballage d'origine (retour gratuit en point relais Djassa), remboursement sous 5 jours ouvrés sur le moyen de paiement utilisé.


In [36]:
print("════ Version LangGraph ════")
afficher_messages(agent_lg.invoke({"messages": [("user", CLIENT)]})["messages"])

════ Version LangGraph ════


Appelle      │ verifier_stock({'produit': 'iPhone 16', 'couleur': 'rose', 'taille': ''})
Appelle      │ fiche_produit({'produit': 'iPhone 16'})
Appelle      │ delai_livraison({'ville': 'Cocody'})
Appelle      │ infos_pratiques({})
Reçoit       │ iphone 16 rose : rupture de stock. Couleurs disponibles : noir, blanc, turquoise.
Reçoit       │ IPHONE 16 (Apple) : 690 000 FCFA au lieu de 730 000 FCFA
Écran 6,1 pouces, puce A18, appareil photo 48 MP et USB-C. Idéal pour la photo et les réseaux sociaux.
Caractéristiques : stockage : 128 Go, écran : 6,1 pouces, photo : 48 MP, garantie : 2 ans
Couleurs en stock : noir, blanc, turquoise. Note : 4.7/5 (328 avis).
  « Très bon téléphone, rapide et excellent en photo. » (Fatou, 5/5)
  « Très agréable à utiliser, mais le prix reste élevé. » (Kouamé, 4/5)
Reçoit       │ Standard : jeudi 29/10 (offerte dès 50 000 FCFA, sinon 1 500 FCFA). Express : mercredi 28/10 (2 500 FCFA).
Reçoit       │ Retours : Retour sous 14 jours, article non utilisé et dans 

Dans la trace LangGraph, plusieurs lignes Appelle arrivent **d'un coup** avant les lignes Reçoit : le LLM peut demander plusieurs outils **dans le même tour** (stock, livraison, retours). Le programme les exécute tous, puis lui renvoie tous les résultats.

<p align="center"><img src="assets/diagrams/nb2_scenario_complet.png" alt="Un message client de quatre questions déclenche quatre appels d'outils dans le même tour, puis une seule réponse" width="860"></p>

**Python pur et LangGraph ne donnent pas toujours exactement la même réponse** : le LLM n'est pas déterministe.

Si tu essaies `FOURNISSEUR = "ollama"` à l'étape 0, tu verras qu'un petit modèle local comme qwen2.5:7b **oublie parfois un outil**, par exemple la livraison : il répond alors sans avoir vérifié. C'est une vraie leçon : **la fiabilité d'un agent dépend du modèle**, mais aussi de la clarté des descriptions d'outils, du prompt, des données et du traitement des erreurs.

### Teste tes propres questions

Modifie la phrase ci-dessous et relance la cellule. Quelques idées :
- *« Le hoodie Adidas beige en M, vous l'avez ? »* : la taille est en rupture, Awa propose-t-elle une autre taille ?
- *« Les AirPods 4 valent le coup ? Que disent les clients ? »*
- *« Qu'est-ce que vous avez en promo en ce moment ? »*
- *« Vous vendez des PlayStation 5 ? »* : le produit n'est pas au catalogue, Awa invente-t-elle ?
- *« Vous avez les Nike Air Max en 42 ? »* : Awa les confond-elle avec les Air Force 1 ?

In [37]:
ma_question = "Le hoodie Adidas beige en M, vous l'avez ?"
afficher_messages(agent_lg.invoke({"messages": [("user", ma_question)]})["messages"])

Appelle      │ verifier_stock({'produit': 'hoodie Adidas', 'couleur': 'beige', 'taille': 'M'})
Reçoit       │ hoodie adidas essentials beige : taille 'M' indisponible. Tailles en stock : S, L, XL.

Répond       │ Bonjour — Awa de Djassa ici. Le hoodie Adidas beige en taille M est actuellement indisponible ; seules les tailles S, L et XL sont en stock. Voulez-vous que je recherche ce même modèle en une autre couleur en M ou que je cherche d'autres hoodies Adidas disponibles en taille M ?


---
## Synthèse

**Les 4 idées à garder**
1. Le LLM ne connaît pas ta boutique : **les outils lui donnent accès aux vraies données**.
2. Le LLM **ne fait que demander**. Ton code exécute, donc tu gardes le contrôle.
3. Ce que le LLM envoie **se valide** (Pydantic) avant d'être exécuté, et les actions à risque demandent d'autres garde-fous.
4. Les frameworks **ne font rien de magique** : ils automatisent la boucle que tu as écrite à la main.

**Repères**

| Concept | Python pur | Framework |
|---|---|---|
| Outil | `def` + fiche JSON (`decrire`) | `tool(fonction)` (LangChain) |
| Tool calling | `tools=SCHEMAS` | `llm.bind_tools()` (LangChain) |
| Boucle ReAct | boucle `for` | `StateGraph` + `ToolNode` (LangGraph) |
| Validation | `executer()` + Pydantic | `tool(..., args_schema=...)` |
| Agent | `agent_python()` | `agent_lg` |

**Prochain notebook** : Awa va lire des documents (livraison, retours, garantie, paiement, guide des tailles) pour répondre à des questions plus ouvertes. C'est le **RAG**.